# 🔬 Actividad Final — Clasificación de Lesiones de Piel con CNN
### Dataset: HAM10000 · Transfer Learning · Ejecución 100% local

**Curso:** PFAD-PADCC-VIC04 · Visión por Computadora · TecNM Virtual 2025

---

## ¿Qué aprenderás con este notebook?

Entrenas una red neuronal convolucional para **clasificar automáticamente
7 tipos de lesiones cutáneas** a partir de imágenes dermoscópicas reales.

Este es un problema de **visión médica asistida por IA**: el modelo aprende
a distinguir lesiones benignas de malignas con la misma lógica que usaste
en los Temas 1, 2 y 3 del curso, pero aplicada a un dataset clínico real.

---

## Las 7 clases de HAM10000

| Clave | Nombre completo | Tipo | Riesgo |
|-------|----------------|------|--------|
| `mel` | Melanoma | Maligno | 🔴 Alto |
| `nv` | Melanocytic nevi (lunar común) | Benigno | 🟢 Bajo |
| `bcc` | Basal cell carcinoma | Maligno | 🟡 Medio-alto |
| `akiec` | Actinic keratoses / Intraepithelial carcinoma | Pre-maligno | 🟡 Medio |
| `bkl` | Benign keratosis-like lesions | Benigno | 🟢 Bajo |
| `df` | Dermatofibroma | Benigno | 🟢 Bajo |
| `vasc` | Vascular lesions | Benigno | 🟢 Bajo |

> ⚠️ **Nota importante:** Este modelo es un ejercicio académico.
> No está validado clínicamente y **no debe usarse para diagnóstico médico real**.

---

## Descarga del dataset

1. Ve a: https://www.kaggle.com/datasets/kmader/skin-cancer-mnist-ham10000
2. Descarga el ZIP completo (~2 GB)
3. Pon la ruta en `ZIP_PATH` de la **Celda 0**

El ZIP de Kaggle contiene:
```
HAM10000_images_part_1/   ← ~5000 imágenes .jpg
HAM10000_images_part_2/   ← ~5000 imágenes .jpg
HAM10000_metadata.csv     ← etiquetas: image_id, dx, age, sex, localization
```

---

## Archivos que genera este notebook

```
proyecto_piel/
├── data/
│   ├── distribucion.png        ← barras con conteo por clase
│   ├── muestras_dataset.png    ← 3 imágenes por clase
│   ├── curvas.png              ← loss / accuracy / F1 por época
│   ├── confusion_counts.png    ← matriz de confusión (conteos)
│   ├── confusion_norm.png      ← matriz de confusión (recall por clase)
│   ├── metricas_clase.png      ← precision/recall/F1 por clase
│   ├── predicciones_grid.png   ← 16 predicciones del test set
│   ├── historial.csv           ← métricas por época
│   └── reporte.csv             ← reporte de clasificación
└── models/
    ├── modelo_best.pth         ← mejor checkpoint durante entrenamiento
    ├── modelo_produccion.pth   ← listo para FastAPI / despliegue
    └── idx2clase.json          ← mapeo 0→melanoma, 1→nevi, etc.
```

---

## Tiempo estimado de entrenamiento

| Hardware | `PC_DEBIL` | Tiempo por época | Total estimado |
|----------|-----------|-----------------|----------------|
| CPU básica (< 8 GB RAM) | `True` | 8–15 min | 1.5–3 h |
| CPU media (≥ 8 GB RAM) | `False` | 15–30 min | 5–10 h |
| GPU dedicada (cualquiera) | `False` | 1–3 min | 20–40 min |

> **Tip:** Si tu PC es lenta, pon `PC_DEBIL = True`. Entrenas menos épocas
> con un modelo más ligero pero el pipeline completo funciona igual.


---
## Celda 0 — Configuración
> **Edita solo esta celda.** El resto del notebook corre automáticamente.

In [ ]:
# =============================================================
# ⚙️  CONFIGURACIÓN — edita solo aquí
# =============================================================

# ─── RUTA DEL ZIP ─────────────────────────────────────────────
# Descarga el ZIP de Kaggle y escribe aquí la ruta completa.
# Ejemplos:
#   Windows : r'C:\Users\maria\Downloads\skin-lesion-analysis.zip'
#   Mac/Linux: '/home/maria/Downloads/skin-lesion-analysis.zip'
ZIP_PATH = r'C:\Users\usuario\Downloads\skin-lesion-analysis.zip'

# ─── CARPETA DE SALIDA ────────────────────────────────────────
# Aquí se guardarán imágenes, modelos y CSV.
# Se crea automáticamente si no existe.
BASE_PATH_STR = r'C:\Users\usuario\proyecto_piel'

# ─── MODO DE HARDWARE ─────────────────────────────────────────
# True  → MobileNetV3-Large, batch=8, imagen 128×128, 10 épocas
#         Usa esto si tu PC tiene < 8 GB de RAM o el entrenamiento es muy lento
# False → EfficientNet-B2, batch=16, imagen 224×224, 20 épocas
#         Usa esto si tienes ≥ 8 GB de RAM o GPU dedicada
PC_DEBIL = False

# ─── HIPERPARÁMETROS AVANZADOS ────────────────────────────────
# Deja todo en None para usar los valores automáticos según PC_DEBIL.
# Cambia solo si sabes lo que estás haciendo.
MODEL_OVERRIDE  = None   # ej: 'mobilenetv3_small_100'
BATCH_OVERRIDE  = None   # ej: 8
IMG_OVERRIDE    = None   # ej: 128
EPOCHS_OVERRIDE = None   # ej: 15

# ─── OTROS (no cambiar sin motivo) ────────────────────────────
LR        = 3e-4   # tasa de aprendizaje inicial
PATIENCE  = 5      # épocas sin mejora → detener entrenamiento
SEED      = 42     # semilla para reproducibilidad
VAL_SIZE  = 0.15   # 15% de las imágenes para validación
TEST_SIZE = 0.15   # 15% de las imágenes para prueba final

# =============================================================
print('✅ Configuración cargada')
print(f'   ZIP        : {ZIP_PATH}')
print(f'   Salida     : {BASE_PATH_STR}')
print(f'   PC débil   : {PC_DEBIL}')


---
## Celda 1 — Instalación de dependencias

Ejecuta esta celda **una sola vez**. Si falla algún paquete, ejecútala de nuevo.

| Paquete | Para qué sirve |
|---------|---------------|
| `torch` + `torchvision` | Framework de deep learning |
| `timm` | Biblioteca de modelos preentrenados (EfficientNet, MobileNet…) |
| `albumentations` | Augmentación de imágenes médicas |
| `torchmetrics` | Accuracy y F1-score durante el entrenamiento |
| `scikit-learn` | Matriz de confusión y reporte de clasificación |
| `seaborn` | Visualización de heatmaps (matrices de confusión) |
| `pandas` | Manejo del CSV de metadatos de HAM10000 |


In [ ]:
import subprocess, sys

paquetes = [
    'torch',
    'torchvision',
    'timm',
    'albumentations',
    'torchmetrics',
    'scikit-learn',
    'seaborn',
    'pandas',
    'Pillow',
]

print('Instalando dependencias...')
for pkg in paquetes:
    r = subprocess.run(
        [sys.executable, '-m', 'pip', 'install', '-q', pkg],
        capture_output=True
    )
    print(f'  {"✅" if r.returncode == 0 else "❌"} {pkg}')

print('\n✅ Listo. Continúa con Celda 2.')


---
## Celda 2 — Imports y configuración del entorno

Esta celda:
- Importa todas las bibliotecas
- Detecta si hay GPU disponible (usa CPU si no hay)
- Fija la semilla aleatoria para que los resultados sean reproducibles
- Crea las carpetas del proyecto
- Define los hiperparámetros según `PC_DEBIL`


In [ ]:
import os, json, shutil, zipfile, random, warnings
from pathlib import Path
import numpy as np
import pandas as pd
from PIL import Image
import matplotlib
# Agg: guarda figuras como PNG sin abrir ventanas emergentes.
# Necesario en entornos locales donde el backend gráfico puede fallar.
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
warnings.filterwarnings('ignore')

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
import timm
import albumentations as A
from albumentations.pytorch import ToTensorV2
from torchmetrics import Accuracy, F1Score
from sklearn.metrics import classification_report, confusion_matrix
from torch.optim.lr_scheduler import CosineAnnealingWarmRestarts

# ── Semilla global ─────────────────────────────────────────────
# Fija el azar en todas las bibliotecas → mismos resultados en cada ejecución
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark     = False

# ── Dispositivo de cómputo ─────────────────────────────────────
# PyTorch detecta automáticamente si hay GPU NVIDIA disponible.
# Si no hay GPU, entrena en CPU (más lento pero funcional).
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'🖥️  Dispositivo : {DEVICE}')
if DEVICE == 'cuda':
    print(f'   GPU         : {torch.cuda.get_device_name(0)}')
    print(f'   VRAM        : {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB')
else:
    print('   CPU mode — el entrenamiento es más lento pero funciona correctamente')
    if PC_DEBIL:
        print('   PC_DEBIL = True → modo ligero activado ✅')
    else:
        print('   Tip: si el entrenamiento tarda demasiado, cambia PC_DEBIL = True')

# ── Hiperparámetros según hardware ────────────────────────────
# PC_DEBIL=True  → modelo pequeño, imágenes pequeñas, pocas épocas
# PC_DEBIL=False → EfficientNet-B2, imágenes 224×224, más épocas
if PC_DEBIL:
    _model  = 'mobilenetv3_large_100'  # 5.4 M parámetros, muy rápido
    _batch  = 8
    _img    = 128
    _epochs = 10
else:
    _model  = 'efficientnet_b2'        # 9.1 M parámetros, mejor accuracy
    _batch  = 16
    _img    = 224
    _epochs = 20

MODEL_NAME = MODEL_OVERRIDE  or _model
BATCH_SIZE = BATCH_OVERRIDE  or _batch
IMG_SIZE   = IMG_OVERRIDE    or _img
EPOCHS     = EPOCHS_OVERRIDE or _epochs
# NUM_WORKERS = 0 siempre en local.
# En Windows, usar workers > 0 con DataLoader causa errores de multiprocessing.
NUM_WORKERS = 0

# Estadísticas de normalización de ImageNet.
# Se usan porque el backbone fue preentrenado con estas cifras.
# Normalizar con los mismos valores garantiza que las activaciones
# del backbone tengan la escala correcta desde el inicio.
MEAN = [0.485, 0.456, 0.406]
STD  = [0.229, 0.224, 0.225]

# ── Carpetas del proyecto ──────────────────────────────────────
BASE_PATH   = Path(BASE_PATH_STR)
IMAGES_PATH = BASE_PATH / 'images'
ZIP_TMP     = BASE_PATH / 'tmp_zip'
for d in ['data', 'images', 'models', 'tmp_zip']:
    (BASE_PATH / d).mkdir(parents=True, exist_ok=True)

print(f'\n📂 Proyecto en: {BASE_PATH}')
print(f'\n⚙️  Hiperparámetros:')
print(f'   Modelo     : {MODEL_NAME}')
print(f'   Batch size : {BATCH_SIZE}')
print(f'   Img size   : {IMG_SIZE}×{IMG_SIZE} px')
print(f'   Épocas máx : {EPOCHS}')
print(f'   Patience   : {PATIENCE}')
print(f'   LR inicial : {LR}')


---
## Celda 3 — Descompresión del ZIP

HAM10000 viene del Kaggle con esta estructura interna:

```
skin-lesion-analysis.zip
├── HAM10000_images_part_1/
│   ├── ISIC_0024306.jpg
│   ├── ISIC_0024307.jpg
│   └── ... (~5000 imágenes)
├── HAM10000_images_part_2/
│   └── ... (~5000 imágenes)
└── HAM10000_metadata.csv    ← etiquetas de diagnóstico
```

La siguiente celda descomprime todo al directorio temporal `tmp_zip/`.


In [ ]:
zip_path = Path(ZIP_PATH)

# Verificar que el ZIP existe antes de intentar abrirlo
if not zip_path.exists():
    raise FileNotFoundError(
        f'\n❌ ZIP no encontrado: {zip_path}'
        f'\n   Verifica que ZIP_PATH en Celda 0 sea correcto.'
        f'\n   Descarga el dataset en: '
        f'https://www.kaggle.com/datasets/kmader/'
        f'skin-lesion-analysis-toward-melanoma-detection'
    )

tam_mb = zip_path.stat().st_size / 1e6
print(f'📦 Descomprimiendo {zip_path.name} ({tam_mb:.0f} MB)...')
print('   Esto puede tardar 2–5 minutos según la velocidad del disco.')

with zipfile.ZipFile(zip_path, 'r') as zf:
    miembros = zf.namelist()
    print(f'   Archivos en el ZIP: {len(miembros):,}')
    zf.extractall(str(ZIP_TMP))

print(f'✅ Descompresión completada → {ZIP_TMP}')


---
## Celda 4 — Organización del dataset por clases

HAM10000 **no** viene organizado en carpetas por clase.
Viene como 10,015 imágenes sueltas + un CSV con los diagnósticos.

Esta celda:
1. Busca `HAM10000_metadata.csv` dentro del ZIP
2. Lee la columna `dx` (diagnóstico abreviado: `mel`, `nv`, `bcc`…)
3. Traduce cada abreviatura a un nombre legible (`melanoma`, `nevi`…)
4. Copia cada imagen a `images/train|val|test/nombre_clase/`
5. Divide automáticamente en 70% train / 15% val / 15% test

El mapeo completo de abreviaturas:

| Columna `dx` | Carpeta generada |
|-------------|-----------------|
| `mel` | `melanoma` |
| `nv` | `melanocytic_nevi` |
| `bcc` | `basal_cell_carcinoma` |
| `akiec` | `actinic_keratoses` |
| `bkl` | `benign_keratosis` |
| `df` | `dermatofibroma` |
| `vasc` | `vascular_lesions` |


In [ ]:
# Mapeo dx → nombre de carpeta legible
DX_MAP = {
    'mel':   'melanoma',
    'nv':    'melanocytic_nevi',
    'bcc':   'basal_cell_carcinoma',
    'akiec': 'actinic_keratoses',
    'bkl':   'benign_keratosis',
    'df':    'dermatofibroma',
    'vasc':  'vascular_lesions',
}

TRAIN_DIR = IMAGES_PATH / 'train'
VAL_DIR   = IMAGES_PATH / 'val'
TEST_DIR  = IMAGES_PATH / 'test'

# ── 1. Buscar el CSV de metadatos ─────────────────────────────
print('🔍 Buscando HAM10000_metadata.csv...')
meta_csv = None
for csv_file in ZIP_TMP.rglob('*.csv'):
    try:
        df_tmp = pd.read_csv(csv_file)
        # El CSV correcto tiene las columnas 'dx' e 'image_id'
        if 'dx' in df_tmp.columns and 'image_id' in df_tmp.columns:
            meta_csv = csv_file
            break
    except Exception:
        continue

if meta_csv is None:
    raise FileNotFoundError(
        '❌ No se encontró HAM10000_metadata.csv dentro del ZIP.\n'
        '   Asegúrate de descargar el ZIP completo de Kaggle (no solo las imágenes).'
    )

print(f'   ✅ CSV encontrado: {meta_csv.name}')
df_meta = pd.read_csv(meta_csv)
print(f'   Registros       : {len(df_meta):,}')
print(f'   Columnas        : {list(df_meta.columns)}')
print(f'   Diagnósticos únicos: {sorted(df_meta["dx"].unique())}')

# ── 2. Traducir dx → nombre de clase ──────────────────────────
df_meta['clase'] = df_meta['dx'].map(DX_MAP)
clases_faltantes = df_meta[df_meta['clase'].isna()]['dx'].unique()
if len(clases_faltantes):
    print(f'   ⚠️  Diagnósticos sin mapeo (se omiten): {clases_faltantes}')
df_meta = df_meta.dropna(subset=['clase'])

# ── 3. Indexar todas las imágenes del ZIP ─────────────────────
print('\n🔍 Indexando imágenes...')
imgs_index = {}   # image_id → Path
for img in ZIP_TMP.rglob('*.jpg'):
    imgs_index[img.stem] = img
for img in ZIP_TMP.rglob('*.JPG'):
    imgs_index[img.stem] = img
print(f'   Imágenes encontradas: {len(imgs_index):,}')

# Verificar cobertura
sin_imagen = df_meta[~df_meta['image_id'].isin(imgs_index)]['image_id']
if len(sin_imagen):
    print(f'   ⚠️  {len(sin_imagen)} registros del CSV sin imagen correspondiente (se omiten)')
df_meta = df_meta[df_meta['image_id'].isin(imgs_index)]
print(f'   Registros usables   : {len(df_meta):,}')

# ── 4. Dividir y copiar por clase ─────────────────────────────
print('\n📁 Organizando en train / val / test...')
print(f'   Proporción: {int((1-VAL_SIZE-TEST_SIZE)*100)}% train / '
      f'{int(VAL_SIZE*100)}% val / {int(TEST_SIZE*100)}% test')

resumen = {}
for clase, grupo in df_meta.groupby('clase'):
    imgs = [imgs_index[iid] for iid in grupo['image_id'] if iid in imgs_index]
    random.seed(SEED)
    random.shuffle(imgs)

    n     = len(imgs)
    n_val = max(1, int(n * VAL_SIZE))
    n_tst = max(1, int(n * TEST_SIZE))
    n_tr  = n - n_val - n_tst

    splits = {
        'train': imgs[:n_tr],
        'val':   imgs[n_tr : n_tr + n_val],
        'test':  imgs[n_tr + n_val :],
    }
    for sname, simgs in splits.items():
        dst = IMAGES_PATH / sname / clase
        dst.mkdir(parents=True, exist_ok=True)
        for img in simgs:
            shutil.copy2(str(img), str(dst / img.name))

    resumen[clase] = {'train': n_tr, 'val': n_val, 'test': n_tst, 'total': n}
    print(f'   {clase:<30} total={n:>5}  train={n_tr:>4}  val={n_val:>3}  test={n_tst:>3}')

# ── 5. Limpiar temporal ───────────────────────────────────────
shutil.rmtree(str(ZIP_TMP), ignore_errors=True)
print('\n🗑️  Carpeta temporal eliminada')

# ── 6. Verificación ───────────────────────────────────────────
print('\n✅ Verificación final:')
total = 0
for split in ['train', 'val', 'test']:
    sp = IMAGES_PATH / split
    clases_sp = [p for p in sp.iterdir() if p.is_dir()]
    n = sum(len(list(c.glob('*.jpg'))) for c in clases_sp)
    total += n
    print(f'   {split:<6}: {len(clases_sp)} clases · {n:,} imágenes')
print(f'   Total : {total:,} imágenes')
print('\n🟢 Dataset listo. Continúa con Celda 5.')


---
## Celda 4b — Diagnóstico (solo si Celda 4 falló)

Muestra el árbol de carpetas del ZIP para identificar qué salió mal.

In [ ]:
# Ejecuta esta celda SOLO si la Celda 4 falló y quieres ver la estructura del ZIP
def arbol(path, nivel=0, max_n=3, max_i=8):
    if nivel > max_n: return
    try:
        items = sorted(path.iterdir())[:max_i]
    except Exception: return
    for item in items:
        pre = '  ' * nivel
        if item.is_dir():
            n = len(list(item.rglob('*.jpg')))
            print(f'{pre}📂 {item.name}/  ({n} imágenes .jpg)')
            arbol(item, nivel+1, max_n, max_i)
        elif item.suffix.lower() in ['.csv','.txt','.json']:
            print(f'{pre}📄 {item.name}')

# Volver a descomprimir si ya se limpió
if not ZIP_TMP.exists() or not any(ZIP_TMP.iterdir()):
    ZIP_TMP.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(Path(ZIP_PATH), 'r') as zf:
        zf.extractall(str(ZIP_TMP))

print('Árbol del ZIP descomprimido:')
print('='*50)
arbol(ZIP_TMP)
print('\nComparte este árbol con el facilitador si necesitas ayuda.')


---
## Celda 5 — Análisis visual del dataset

HAM10000 tiene un **gran desbalance de clases**: la clase `nv` (lunares comunes)
tiene ~6,700 imágenes mientras que `df` (dermatofibroma) tiene solo ~115.

Esto es un problema real en datos médicos: las enfermedades raras tienen
menos ejemplos disponibles. Por eso en Celda 6 usamos `WeightedRandomSampler`
para compensar ese desbalance durante el entrenamiento.


In [ ]:
# =============================================================
# 📊 INVENTARIO DE CLASES
# =============================================================
clases_train = sorted([p.name for p in (IMAGES_PATH/'train').iterdir() if p.is_dir()])
NUM_CLASES   = len(clases_train)
clase2idx    = {c: i for i, c in enumerate(clases_train)}
idx2clase    = {str(i): c for c, i in clase2idx.items()}

print(f'Clases ({NUM_CLASES}):')
conteos = {}
for split in ['train', 'val', 'test']:
    sp = IMAGES_PATH / split
    if not sp.exists(): continue
    for cd in sp.iterdir():
        if not cd.is_dir(): continue
        n = len(list(cd.glob('*.jpg')))
        conteos.setdefault(cd.name, {})[split] = n

df_dist = pd.DataFrame(conteos).T.fillna(0).astype(int)
df_dist['total'] = df_dist.sum(axis=1)
df_dist = df_dist.sort_values('total', ascending=False)
print(df_dist.to_string())

with open(BASE_PATH/'data'/'idx2clase.json','w',encoding='utf-8') as f:
    json.dump(idx2clase, f, ensure_ascii=False, indent=2)
print(f'\n✅ idx2clase.json guardado')


In [ ]:
# =============================================================
# 📊 GRÁFICA DE DISTRIBUCIÓN DE CLASES
# Nota: HAM10000 tiene desbalance severo (nv >> el resto).
# La barra roja indica < 200 imágenes (clase minoritaria).
# =============================================================
fig, axes = plt.subplots(1, 2, figsize=(16, 5))
fig.suptitle('HAM10000 — Distribución de clases (7 tipos de lesiones)',
             fontsize=13, fontweight='bold')

colores = ['#e74c3c' if v < 200 else '#f39c12' if v < 1000 else '#27ae60'
           for v in df_dist['total'].values]
axes[0].barh(df_dist.index, df_dist['total'], color=colores)
axes[0].set_xlabel('Imágenes totales')
axes[0].set_title('Distribución por clase')
for i, v in enumerate(df_dist['total']):
    axes[0].text(v + 20, i, str(v), va='center', fontsize=9)
parches = [mpatches.Patch(color='#e74c3c', label='< 200 (minoritaria)'),
           mpatches.Patch(color='#f39c12', label='200–1000'),
           mpatches.Patch(color='#27ae60', label='> 1000 (mayoritaria)')]
axes[0].legend(handles=parches, fontsize=9)

splits_tot = {s: df_dist[s].sum() for s in ['train','val','test'] if s in df_dist.columns}
axes[1].pie(splits_tot.values(), labels=splits_tot.keys(),
            autopct='%1.1f%%', colors=['#3498db','#e67e22','#2ecc71'],
            startangle=90, textprops={'fontsize':11})
axes[1].set_title('Proporción train / val / test')

plt.tight_layout()
out = BASE_PATH / 'data' / 'distribucion.png'
plt.savefig(out, dpi=150, bbox_inches='tight')
plt.close()
print(f'📁 distribucion.png guardado → {out}')


In [ ]:
# =============================================================
# 🖼️  MUESTRAS DEL DATASET — 3 imágenes por clase
# Las imágenes de dermoscopia muestran la piel aumentada.
# Nota cómo melanoma tiene bordes irregulares y colores variados.
# =============================================================
COLS = 3
fig, axes = plt.subplots(NUM_CLASES, COLS, figsize=(COLS * 3, NUM_CLASES * 3))
fig.suptitle('HAM10000 — 3 muestras por clase (imágenes dermoscópicas)',
             fontsize=12, fontweight='bold')

for fila, clase in enumerate(clases_train):
    imgs = list((IMAGES_PATH / 'train' / clase).glob('*.jpg'))
    random.shuffle(imgs)
    for col in range(COLS):
        ax = axes[fila][col]
        if col < len(imgs):
            try:
                ax.imshow(Image.open(imgs[col]).convert('RGB'))
            except Exception:
                pass
        ax.axis('off')
        if col == 0:
            ax.set_ylabel(clase.replace('_',' '), fontsize=8,
                          rotation=0, labelpad=90, va='center')

plt.tight_layout()
out = BASE_PATH / 'data' / 'muestras_dataset.png'
plt.savefig(out, dpi=150, bbox_inches='tight')
plt.close()
print(f'📁 muestras_dataset.png guardado → {out}')


---
## Celda 6 — Augmentación y DataLoaders

### ¿Por qué augmentación especial para imágenes médicas?

Las imágenes dermoscópicas tienen características únicas:
- La **orientación** de la lesión no importa (puede estar girada cualquier ángulo)
- El **brillo** varía según el dispositivo y la piel del paciente
- Los **artefactos** (pelos, burbujas, reglas de medición) son parte del dataset real

Por eso usamos augmentaciones que simulan estas variaciones:

| Transformación | Propósito |
|---------------|-----------|
| `HorizontalFlip` + `VerticalFlip` | La orientación de la lesión es irrelevante |
| `RandomRotate90` | Cualquier ángulo es igualmente probable |
| `CLAHE` | Mejora contraste → más detalle en lesiones oscuras |
| `ColorJitter` | Simula variaciones de iluminación entre dispositivos |
| `GaussNoise` | Simula el ruido del sensor de la cámara dermoscópica |
| `CoarseDropout` | Simula oclusiones por artefactos (pelos, marcas) |

### WeightedRandomSampler — compensar el desbalance

HAM10000 tiene ~6,700 imágenes de `nv` pero solo ~115 de `df`.
Sin balanceo, el modelo aprendería a predecir siempre `nv` y tendría
accuracy alta pero sería inútil para detectar enfermedades raras.

`WeightedRandomSampler` asigna mayor probabilidad de selección a las
clases minoritarias, equilibrando el entrenamiento artificialmente.


In [ ]:
# =============================================================
# 🎨 AUGMENTACIÓN ADAPTADA AL HARDWARE Y AL DOMINIO MÉDICO
# =============================================================
if PC_DEBIL:
    # Augmentación ligera → menos tiempo de CPU, menos RAM
    train_aug = A.Compose([
        A.Resize(IMG_SIZE, IMG_SIZE),
        A.HorizontalFlip(p=0.5),
        A.VerticalFlip(p=0.5),
        A.RandomRotate90(p=0.5),
        A.ColorJitter(brightness=0.15, contrast=0.15, p=0.4),
        A.Normalize(mean=MEAN, std=STD),
        ToTensorV2(),
    ])
    print('[PC_DEBIL] Augmentación ligera')
else:
    # Augmentación completa → mejor generalización en imágenes médicas
    train_aug = A.Compose([
        A.Resize(IMG_SIZE, IMG_SIZE),
        A.HorizontalFlip(p=0.5),
        A.VerticalFlip(p=0.5),          # válido: lesiones pueden estar en cualquier orientación
        A.RandomRotate90(p=0.5),
        A.ShiftScaleRotate(shift_limit=0.05, scale_limit=0.1,
                           rotate_limit=20, p=0.5),
        A.ColorJitter(brightness=0.25, contrast=0.25,
                      saturation=0.15, hue=0.05, p=0.5),
        A.CLAHE(clip_limit=3.0, p=0.4),      # mejora contraste en lesiones oscuras
        A.GaussianBlur(blur_limit=(3, 5), p=0.2),
        A.GaussNoise(var_limit=(5, 25), p=0.25),
        A.CoarseDropout(max_holes=4, max_height=24,  # simula oclusión por artefactos
                        max_width=24, p=0.25),
        A.Normalize(mean=MEAN, std=STD),
        ToTensorV2(),
    ])
    print('[Normal] Augmentación completa para imágenes médicas')

# Validación/test: SOLO resize y normalización.
# No se augmenta: queremos evaluar con las imágenes tal como son.
val_aug = A.Compose([
    A.Resize(IMG_SIZE, IMG_SIZE),
    A.Normalize(mean=MEAN, std=STD),
    ToTensorV2(),
])

# =============================================================
# 📂 DATASET PERSONALIZADO
# =============================================================
class HAMDataset(Dataset):
    """
    Lee imágenes de la estructura images/split/clase/imagen.jpg
    y aplica las transformaciones de albumentations.

    Manejo de errores: si una imagen está corrupta, devuelve
    un tensor negro en lugar de abortar el entrenamiento.
    """
    def __init__(self, split_path, clase2idx, transform=None):
        self.samples   = []
        self.transform = transform
        for clase_dir in sorted(split_path.iterdir()):
            if not clase_dir.is_dir():
                continue
            idx = clase2idx.get(clase_dir.name)
            if idx is None:
                continue
            for img_path in clase_dir.glob('*.jpg'):
                self.samples.append((img_path, idx))

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, i):
        path, label = self.samples[i]
        try:
            img = np.array(Image.open(path).convert('RGB'))
        except Exception:
            img = np.zeros((IMG_SIZE, IMG_SIZE, 3), dtype=np.uint8)
        if self.transform:
            img = self.transform(image=img)['image']
        return img, torch.tensor(label, dtype=torch.long)


train_ds = HAMDataset(IMAGES_PATH / 'train', clase2idx, train_aug)
val_ds   = HAMDataset(IMAGES_PATH / 'val',   clase2idx, val_aug)
test_ds  = HAMDataset(IMAGES_PATH / 'test',  clase2idx, val_aug)

# =============================================================
# ⚖️  WEIGHTED RANDOM SAMPLER — balanceo de clases
# =============================================================
# Para cada imagen en train, calculamos su peso = 1 / frecuencia_de_su_clase.
# Las clases con pocas imágenes (df, vasc) tienen peso alto → se muestrean más.
# Las clases con muchas imágenes (nv) tienen peso bajo → se muestrean menos.
etiquetas_train = [s[1] for s in train_ds.samples]
conteo_clases   = [etiquetas_train.count(i) for i in range(NUM_CLASES)]
pesos_muestras  = [1.0 / max(conteo_clases[lbl], 1) for lbl in etiquetas_train]
sampler         = WeightedRandomSampler(pesos_muestras, len(train_ds), replacement=True)

# NUM_WORKERS=0: necesario en Windows local para evitar errores de multiprocessing
train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, sampler=sampler,
                          num_workers=0, pin_memory=(DEVICE=='cuda'))
val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False,
                          num_workers=0, pin_memory=(DEVICE=='cuda'))
test_loader  = DataLoader(test_ds,  batch_size=BATCH_SIZE, shuffle=False,
                          num_workers=0, pin_memory=(DEVICE=='cuda'))

print('✅ DataLoaders listos')
print(f'   train : {len(train_ds):,} imágenes · {len(train_loader)} batches')
print(f'   val   : {len(val_ds):,} imágenes · {len(val_loader)} batches')
print(f'   test  : {len(test_ds):,} imágenes · {len(test_loader)} batches')
print(f'   WeightedSampler activo — balancea desigualdad entre las 7 clases')


---
## Celda 7 — Modelo CNN con Transfer Learning

### ¿Por qué Transfer Learning y no entrenar desde cero?

HAM10000 tiene ~10,000 imágenes. Entrenar una CNN desde cero con tan pocas
imágenes generaría overfitting severo.

EfficientNet-B2 fue preentrenado con **ImageNet** (1.2 millones de imágenes,
1000 categorías). Sus capas ya saben detectar bordes, texturas, formas y
partes de objetos. Solo reemplazamos la cabeza de clasificación por una
adaptada a nuestras 7 clases de lesiones.

### Arquitectura de la cabeza de clasificación

```
backbone (EfficientNet-B2 congelado parcialmente)
    ↓ vector de 1408 características
Dropout(0.5)     ← apaga 50% de neuronas → evita overfitting
Linear(1408→512) ← combina las características
GELU()           ← activación suave (mejor que ReLU en transformers y CNNs modernas)
Dropout(0.4)
Linear(512→7)    ← una salida por clase
    ↓ logits (no probabilidades aún)
    → CrossEntropyLoss aplica Softmax internamente durante el entrenamiento
    → F.softmax() en inferencia para obtener probabilidades
```


In [ ]:
# =============================================================
# 🧠 MODELO — EfficientNet-B2 / MobileNetV3 + cabeza para HAM10000
# =============================================================
class ClasificadorPiel(nn.Module):
    """
    CNN para clasificación de lesiones cutáneas (HAM10000).

    Backbone: EfficientNet-B2 o MobileNetV3 preentrenado en ImageNet.
    Cabeza  : Dropout → Linear → GELU → Dropout → Linear(NUM_CLASES)

    timm.create_model con num_classes=0 devuelve el backbone sin su
    clasificador original, listo para agregar uno personalizado.
    global_pool='avg' aplica GlobalAveragePooling al final del backbone,
    convirtiendo el mapa de características 3D en un vector 1D.
    """
    def __init__(self, num_classes, model_name=MODEL_NAME, pretrained=True):
        super().__init__()
        self.backbone = timm.create_model(
            model_name,
            pretrained=pretrained,
            num_classes=0,
            global_pool='avg',
        )
        in_f = self.backbone.num_features
        self.classifier = nn.Sequential(
            nn.Dropout(0.5),
            nn.Linear(in_f, min(512, in_f)),
            nn.GELU(),
            nn.Dropout(0.4),
            nn.Linear(min(512, in_f), num_classes),
        )

    def forward(self, x):
        feats = self.backbone(x)       # extracción de características
        return self.classifier(feats)  # clasificación


model     = ClasificadorPiel(NUM_CLASES).to(DEVICE)
total_p   = sum(p.numel() for p in model.parameters())
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)

print(f'✅ Modelo: {MODEL_NAME}')
print(f'   Parámetros totales     : {total_p/1e6:.1f} M')
print(f'   Parámetros entrenables : {trainable/1e6:.1f} M')
print(f'   Clases de salida       : {NUM_CLASES}')

# Verificar que el forward pass no da error antes de entrenar
with torch.no_grad():
    dummy = torch.randn(2, 3, IMG_SIZE, IMG_SIZE).to(DEVICE)
    out   = model(dummy)
    print(f'   Forward pass OK        : {list(dummy.shape)} → {list(out.shape)}')
del dummy, out
if DEVICE == 'cuda':
    torch.cuda.empty_cache()


---
## Celda 8 — Entrenamiento

### Decisiones de diseño explicadas

**CrossEntropyLoss con `weight`:**
Cada clase recibe un peso inversamente proporcional a su frecuencia.
`nv` (la más frecuente) tiene peso bajo; `df` y `vasc` (las minoritarias) tienen peso alto.
Esto penaliza más los errores en clases raras, donde equivocarse tiene mayor costo clínico.

**`label_smoothing=0.1`:**
En lugar de etiquetas duras (0 o 1), usa 0.05 y 0.95.
Evita que el modelo sea demasiado confiado → mejor generalización.

**CosineAnnealingWarmRestarts:**
La tasa de aprendizaje baja en curva cosenoidal y se reinicia periódicamente.
Permite escapar de mínimos locales y converger más suave que un schedule fijo.

**Early Stopping con `PATIENCE=5`:**
Si la accuracy de validación no mejora en 5 épocas seguidas, el entrenamiento
se detiene y se restauran los pesos del mejor checkpoint.


In [ ]:
# =============================================================
# 🚀 ENTRENAMIENTO — HAM10000
# =============================================================

# ── Pesos de clase para CrossEntropyLoss ──────────────────────
# Fórmula: peso_clase_i = N_total / (N_clases × N_imágenes_clase_i)
# Clases con pocas imágenes → peso alto → el modelo las prioriza
cc = [etiquetas_train.count(i) for i in range(NUM_CLASES)]
cls_weights = torch.tensor(
    [len(etiquetas_train) / (NUM_CLASES * max(c, 1)) for c in cc],
    dtype=torch.float32,
).to(DEVICE)

print('Pesos de clase (mayor peso = clase más rara):')
for i, (w, n) in enumerate(zip(cls_weights.cpu(), cc)):
    print(f'   {idx2clase[str(i)]:<30} n={n:>5}  peso={w:.3f}')

criterion = nn.CrossEntropyLoss(weight=cls_weights, label_smoothing=0.1)
optimizer = optim.AdamW(model.parameters(), lr=LR, weight_decay=1e-3)
scheduler = CosineAnnealingWarmRestarts(optimizer, T_0=10, T_mult=2, eta_min=1e-6)

# Métricas de torchmetrics (calculan acumulativamente por época)
acc_m = Accuracy(task='multiclass', num_classes=NUM_CLASES).to(DEVICE)
f1_m  = F1Score(task='multiclass', num_classes=NUM_CLASES, average='macro').to(DEVICE)

MODEL_SAVE = BASE_PATH / 'models' / 'modelo_best.pth'


def train_epoch():
    """Una época: forward + loss + backward + actualización de pesos."""
    model.train()
    total_loss = 0.0
    acc_m.reset()
    for imgs, labels in train_loader:
        imgs   = imgs.to(DEVICE, non_blocking=True)
        labels = labels.to(DEVICE, non_blocking=True)
        optimizer.zero_grad(set_to_none=True)   # más eficiente que zero_grad()
        logits = model(imgs)
        loss   = criterion(logits, labels)
        loss.backward()
        # Clip de gradientes: si el gradiente crece demasiado, lo trunca en 1.0
        # Evita la explosión de gradientes que puede desestabilizar el entrenamiento
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        total_loss += loss.item()
        acc_m.update(logits.detach(), labels)
    return total_loss / len(train_loader), acc_m.compute().item()


@torch.no_grad()
def evaluate(loader):
    """Evaluación sin gradientes (más rápido y sin consumir memoria extra)."""
    model.eval()
    total_loss = 0.0
    acc_m.reset()
    f1_m.reset()
    for imgs, labels in loader:
        imgs   = imgs.to(DEVICE, non_blocking=True)
        labels = labels.to(DEVICE, non_blocking=True)
        logits = model(imgs)
        total_loss += criterion(logits, labels).item()
        acc_m.update(logits, labels)
        f1_m.update(logits, labels)
    return total_loss / len(loader), acc_m.compute().item(), f1_m.compute().item()


# ── Bucle principal ────────────────────────────────────────────
best_acc, no_imp, history = 0.0, 0, []
sep = '─' * 82
print(f'\n🚀 Iniciando entrenamiento — HAM10000')
print(f'   Modelo  : {MODEL_NAME} | Clases: {NUM_CLASES} | Device: {DEVICE}')
print(f'   Épocas  : {EPOCHS} máx | Patience: {PATIENCE}')
print(sep)
print(f'  {"Epoch":>5} | {"TrainLoss":>9} | {"TrainAcc":>8} | '
      f'{"ValLoss":>8} | {"ValAcc":>7} | {"ValF1":>7}')
print(sep)

for epoch in range(1, EPOCHS + 1):
    tl, ta      = train_epoch()
    vl, va, vf1 = evaluate(val_loader)
    scheduler.step(epoch)

    history.append({'epoch':epoch,'train_loss':tl,'train_acc':ta,
                    'val_loss':vl,'val_acc':va,'val_f1':vf1})

    mejoro = va > best_acc
    print(f'  {epoch:5d} | {tl:9.4f} | {ta:8.4f} | '
          f'{vl:8.4f} | {va:7.4f} | {vf1:7.4f}  {"🟢" if mejoro else ""}')

    if mejoro:
        best_acc = va
        no_imp   = 0
        torch.save({
            'epoch': epoch, 'model_state_dict': model.state_dict(),
            'val_acc': va, 'val_f1': vf1, 'idx2clase': idx2clase,
            'num_classes': NUM_CLASES, 'model_name': MODEL_NAME,
            'img_size': IMG_SIZE, 'mean': MEAN, 'std': STD,
            'dataset': 'HAM10000',
        }, MODEL_SAVE)
        print(f'         💾 Checkpoint guardado — val_acc={va:.4f}  val_f1={vf1:.4f}')
    else:
        no_imp += 1
        if no_imp >= PATIENCE:
            print(f'\n⛔ Early stopping en epoch {epoch} — sin mejora por {PATIENCE} épocas')
            break

pd.DataFrame(history).to_csv(BASE_PATH / 'data' / 'historial.csv', index=False)
print(f'\n🏆 Entrenamiento finalizado — Mejor val_acc: {best_acc:.4f} ({best_acc*100:.2f}%)')


---
## Celda 9 — Curvas de entrenamiento

Visualiza cómo evolucionaron loss, accuracy y F1 macro por época.
**Señales de alarma:** si `train_acc` sube pero `val_acc` se estanca → overfitting.

In [ ]:
# =============================================================
# 📈 CURVAS DE ENTRENAMIENTO
# =============================================================
hist_df = pd.DataFrame(history)
best_ep = hist_df.loc[hist_df['val_acc'].idxmax(), 'epoch']

fig, axes = plt.subplots(1, 3, figsize=(16, 4))
fig.suptitle(f'HAM10000 · {MODEL_NAME} · {NUM_CLASES} clases',
             fontsize=12, y=1.02)

for ax, y1, y2, titulo in [
    (axes[0], 'train_loss', 'val_loss',  'Loss'),
    (axes[1], 'train_acc',  'val_acc',   'Accuracy'),
]:
    ax.plot(hist_df['epoch'], hist_df[y1], label='Train', color='#3498db', lw=2)
    ax.plot(hist_df['epoch'], hist_df[y2], label='Val',   color='#e74c3c', lw=2, ls='--')
    ax.set_title(titulo, fontweight='bold')
    ax.set_xlabel('Época')
    ax.legend()
    ax.grid(alpha=0.3)

axes[1].axvline(x=best_ep, color='#27ae60', ls='--', alpha=0.8,
                label=f'Mejor época ({int(best_ep)})')
axes[1].legend()

axes[2].plot(hist_df['epoch'], hist_df['val_f1'], color='#9b59b6', lw=2)
axes[2].fill_between(hist_df['epoch'], hist_df['val_f1'], alpha=0.15, color='#9b59b6')
axes[2].set_title('F1 Macro (Val)', fontweight='bold')
axes[2].set_xlabel('Época')
axes[2].grid(alpha=0.3)

plt.tight_layout()
out = BASE_PATH / 'data' / 'curvas.png'
plt.savefig(out, dpi=150, bbox_inches='tight')
plt.close()
print(f'📁 curvas.png guardado → {out}')


---
## Celda 10 — Evaluación en el conjunto de prueba

Carga el mejor checkpoint y evalúa sobre el test set (imágenes que el modelo **nunca vio** durante el entrenamiento).

In [ ]:
# =============================================================
# 🔬 EVALUACIÓN EN TEST SET
# =============================================================
ckpt = torch.load(MODEL_SAVE, map_location=DEVICE)
model.load_state_dict(ckpt['model_state_dict'])
model.eval()
print(f'✅ Mejor modelo cargado — epoch {ckpt["epoch"]} | val_acc={ckpt["val_acc"]:.4f}')

all_preds, all_labels, all_probs = [], [], []
with torch.no_grad():
    for imgs, labels in test_loader:
        logits = model(imgs.to(DEVICE))
        probs  = F.softmax(logits, dim=1).cpu().numpy()
        preds  = logits.argmax(dim=1).cpu().numpy()
        all_preds.extend(preds)
        all_labels.extend(labels.numpy())
        all_probs.extend(probs)

nombres_clases = [idx2clase[str(i)] for i in range(NUM_CLASES)]
test_acc = (np.array(all_preds) == np.array(all_labels)).mean()

print(f'\n🎯 Test Accuracy : {test_acc:.4f} ({test_acc*100:.2f}%)')
print('\n📊 Reporte de clasificación:')
reporte = classification_report(all_labels, all_preds,
                                 target_names=nombres_clases, digits=3, output_dict=True)
print(classification_report(all_labels, all_preds, target_names=nombres_clases, digits=3))

pd.DataFrame(reporte).T.to_csv(BASE_PATH / 'data' / 'reporte.csv')
print('📁 reporte.csv guardado')


In [ ]:
# =============================================================
# 🟦 MATRICES DE CONFUSIÓN
# =============================================================
# Matriz de conteos: muestra el número absoluto de predicciones por par (real, predicho)
# Matriz normalizada: muestra el recall por clase (qué % de cada clase real fue detectado)
# Para HAM10000 la normalizada es más informativa porque hay desbalance severo.
cm      = confusion_matrix(all_labels, all_preds)
cm_norm = cm.astype('float') / cm.sum(axis=1)[:, np.newaxis]

for fname, data, fmt, cmap, titulo in [
    ('confusion_counts.png', cm,      'd',    'Blues',  'Conteos absolutos'),
    ('confusion_norm.png',   cm_norm, '.2f',  'YlOrRd', 'Normalizada — Recall por clase'),
]:
    fig, ax = plt.subplots(figsize=(9, 7))
    sns.heatmap(data, ax=ax, cmap=cmap, annot=True, fmt=fmt,
                xticklabels=nombres_clases, yticklabels=nombres_clases)
    ax.set_title(f'Matriz de Confusión HAM10000 — {titulo}',
                 fontweight='bold', fontsize=12)
    ax.set_xlabel('Predicho')
    ax.set_ylabel('Real')
    ax.tick_params(axis='x', rotation=45, labelsize=9)
    ax.tick_params(axis='y', rotation=0,  labelsize=9)
    plt.tight_layout()
    out = BASE_PATH / 'data' / fname
    plt.savefig(out, dpi=150, bbox_inches='tight')
    plt.close()
    print(f'📁 {fname} guardado')


In [ ]:
# =============================================================
# 📊 PRECISION / RECALL / F1 POR CLASE
# Para HAM10000, un recall bajo en 'melanoma' es clínicamente grave.
# Un recall bajo en 'nv' (lunar común) es menos preocupante.
# =============================================================
clases_plot = [c for c in nombres_clases if c in reporte]
prec_v = [reporte[c]['precision'] for c in clases_plot]
rec_v  = [reporte[c]['recall']    for c in clases_plot]
f1_v   = [reporte[c]['f1-score']  for c in clases_plot]

x = np.arange(len(clases_plot))
w = 0.25
fig, ax = plt.subplots(figsize=(13, 5))
ax.bar(x - w, prec_v, w, label='Precision', color='#3498db', alpha=0.85)
ax.bar(x,     rec_v,  w, label='Recall',    color='#27ae60', alpha=0.85)
ax.bar(x + w, f1_v,   w, label='F1-Score',  color='#9b59b6', alpha=0.85)
ax.axhline(y=0.5, color='red',   ls='--', alpha=0.4, lw=1, label='Mínimo 0.5')
ax.axhline(y=0.8, color='green', ls='--', alpha=0.4, lw=1, label='Objetivo 0.8')
ax.set_xticks(x)
ax.set_xticklabels([c.replace('_',' ') for c in clases_plot], rotation=30, ha='right')
ax.set_ylim(0, 1.05)
ax.set_ylabel('Score')
ax.set_title('Métricas por clase — HAM10000', fontweight='bold')
ax.legend()
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
out = BASE_PATH / 'data' / 'metricas_clase.png'
plt.savefig(out, dpi=150, bbox_inches='tight')
plt.close()
print(f'📁 metricas_clase.png guardado → {out}')


---
## Celda 11 — Visualización de predicciones

Ve 16 imágenes del test set con la predicción del modelo. Verde = correcto, rojo = error.

In [ ]:
# =============================================================
# 🖼️  GRID DE 16 PREDICCIONES ALEATORIAS DEL TEST SET
# =============================================================
model.eval()
idxs = random.sample(range(len(test_ds)), min(16, len(test_ds)))

fig, axes = plt.subplots(2, 8, figsize=(22, 7))
fig.suptitle('HAM10000 — Predicciones del test set (verde=correcto · rojo=error)',
             fontsize=12, fontweight='bold')

for ax, idx in zip(axes.ravel(), idxs):
    path, lbl_real = test_ds.samples[idx]
    try:
        img_np = np.array(Image.open(path).convert('RGB'))
        tensor = val_aug(image=img_np)['image'].unsqueeze(0).to(DEVICE)
        with torch.no_grad():
            probs    = F.softmax(model(tensor), dim=1)[0]
            pred     = probs.argmax().item()
            conf     = probs[pred].item() * 100
        ax.imshow(img_np)
        ok    = pred == lbl_real
        color = '#27ae60' if ok else '#e74c3c'
        ax.set_title(
            f'{"✅" if ok else "❌"}\n'
            f'{idx2clase[str(pred)].replace("_"," ")[:16]}\n'
            f'{conf:.0f}%',
            fontsize=6.5, color=color
        )
    except Exception:
        pass
    ax.axis('off')

plt.tight_layout()
out = BASE_PATH / 'data' / 'predicciones_grid.png'
plt.savefig(out, dpi=150, bbox_inches='tight')
plt.close()
print(f'📁 predicciones_grid.png guardado → {out}')


In [ ]:
# =============================================================
# 📸 PROBAR CON TU PROPIA IMAGEN
# Pon aquí la ruta a una imagen de lesión cutánea de tu PC.
# El modelo intentará clasificarla entre las 7 clases de HAM10000.
# =============================================================
MI_IMAGEN = r'C:\ruta\a\tu\imagen.jpg'

img_path = Path(MI_IMAGEN)
if not img_path.exists():
    print(f'⚠️  Imagen no encontrada: {img_path}')
    print('   Cambia MI_IMAGEN a la ruta de tu imagen y vuelve a ejecutar esta celda.')
else:
    img_np = np.array(Image.open(img_path).convert('RGB'))
    tensor = val_aug(image=img_np)['image'].unsqueeze(0).to(DEVICE)
    with torch.no_grad():
        probs        = F.softmax(model(tensor), dim=1)[0].cpu()
        top5p, top5i = torch.topk(probs, min(5, NUM_CLASES))

    nombre_pred = idx2clase[str(top5i[0].item())]
    fig, axes = plt.subplots(1, 2, figsize=(12, 5))
    fig.suptitle(f'Tu imagen — Clasificada como: {nombre_pred.replace("_"," ")}',
                 fontsize=12, fontweight='bold')
    axes[0].imshow(img_np)
    axes[0].set_title(img_path.name, fontsize=10)
    axes[0].axis('off')

    top_n = [idx2clase[str(i.item())].replace('_',' ')[:22] for i in top5i]
    top_p = [p.item() * 100 for p in top5p]
    axes[1].barh(top_n[::-1], top_p[::-1], color='#2E6BB0', alpha=0.85, edgecolor='white')
    for i, prob in enumerate(top_p[::-1]):
        axes[1].text(prob + 0.5, i, f'{prob:.1f}%', va='center', fontsize=10)
    axes[1].set_xlim(0, 110)
    axes[1].set_xlabel('Probabilidad (%)')
    axes[1].set_title('Top-5 predicciones')
    axes[1].grid(axis='x', alpha=0.3)
    plt.tight_layout()
    out = BASE_PATH / 'data' / 'mi_prediccion.png'
    plt.savefig(out, dpi=150, bbox_inches='tight')
    plt.close()
    print('🎯 Resultado:')
    for i, (p, idx) in enumerate(zip(top5p, top5i)):
        print(f'   Top-{i+1}: {idx2clase[str(idx.item())].replace("_"," "):<30} {p.item()*100:.2f}%')
    print(f'\n📁 mi_prediccion.png guardado → {out}')


---
## Celda 12 — Exportar modelo para producción

In [ ]:
# =============================================================
# 📦 EXPORTAR MODELO DE PRODUCCIÓN
# =============================================================
ckpt = torch.load(MODEL_SAVE, map_location=DEVICE)
model.load_state_dict(ckpt['model_state_dict'])
model.eval()

prod_path = BASE_PATH / 'models' / 'modelo_produccion.pth'
torch.save({
    'model_state_dict': model.state_dict(),
    'idx2clase':        idx2clase,
    'num_classes':      NUM_CLASES,
    'model_name':       MODEL_NAME,
    'img_size':         IMG_SIZE,
    'mean':             MEAN,
    'std':              STD,
    'val_acc':          ckpt['val_acc'],
    'val_f1':           ckpt['val_f1'],
    'best_epoch':       ckpt['epoch'],
    'dataset':          'HAM10000',
    'clases':           clases_train,
}, prod_path)
print(f'✅ modelo_produccion.pth — {prod_path.stat().st_size/1e6:.1f} MB')

with open(BASE_PATH / 'data' / 'idx2clase.json', 'w', encoding='utf-8') as f:
    json.dump(idx2clase, f, ensure_ascii=False, indent=2)
print(f'✅ idx2clase.json guardado')


---
## Celda 13 — Resumen final

In [ ]:
# =============================================================
# 📋 RESUMEN FINAL — HAM10000
# =============================================================
ckpt = torch.load(MODEL_SAVE, map_location='cpu')
print('=' * 64)
print('  🔬 ACTIVIDAD FINAL — CLASIFICACIÓN DE LESIONES DE PIEL')
print('  Dataset: HAM10000 (7 clases dermoscópicas)')
print('=' * 64)
print(f'  Modelo        : {MODEL_NAME}')
print(f'  PC débil      : {PC_DEBIL}')
print(f'  Dispositivo   : {DEVICE}')
print(f'  Clases        : {NUM_CLASES}')
print(f'  Imágenes      : train={len(train_ds):,} | val={len(val_ds):,} | test={len(test_ds):,}')
print(f'  Mejor época   : {ckpt["epoch"]}')
print(f'  Val Accuracy  : {ckpt["val_acc"]*100:.2f}%')
print(f'  Val F1 Macro  : {ckpt["val_f1"]*100:.2f}%')
print(f'  Test Accuracy : {test_acc*100:.2f}%')
print('=' * 64)

archivos = [
    BASE_PATH / 'models' / 'modelo_best.pth',
    BASE_PATH / 'models' / 'modelo_produccion.pth',
    BASE_PATH / 'data'   / 'idx2clase.json',
    BASE_PATH / 'data'   / 'historial.csv',
    BASE_PATH / 'data'   / 'reporte.csv',
    BASE_PATH / 'data'   / 'curvas.png',
    BASE_PATH / 'data'   / 'distribucion.png',
    BASE_PATH / 'data'   / 'muestras_dataset.png',
    BASE_PATH / 'data'   / 'confusion_counts.png',
    BASE_PATH / 'data'   / 'confusion_norm.png',
    BASE_PATH / 'data'   / 'metricas_clase.png',
    BASE_PATH / 'data'   / 'predicciones_grid.png',
]
print('  Archivos generados:')
for r in archivos:
    ok   = r.exists()
    size = f'{r.stat().st_size/1e6:.1f} MB' if ok else 'no encontrado'
    print(f'  {"✅" if ok else "⚠️"} {r.name:<38} {size}')
print('=' * 64)
